# Elastic Late Chunking Demo

## Create Elastic Serverless Environment

In [ ]:
%%bash
echo "--- Initializing ---"
terraform -chdir=terraform init -upgrade -input=false > /dev/null && echo "Done."
echo "--- Applying Changes ---"
terraform -chdir=terraform apply -auto-approve > /dev/null && echo "Done."

echo "--- Exporting Environment Variables ---"
cat > .env << EOF
ELASTIC_CLOUD_API_KEY=$(terraform -chdir=terraform output -raw elastic_cloud_api_key)
ELASTIC_CLOUD_ID=$(terraform -chdir=terraform output -raw elastic_cloud_id)
JINA_API_KEY=$(terraform -chdir=terraform output -raw jina_api_key)
EOF
echo "Done."

## Create Inference Endpoints

In [ ]:
from dotenv import load_dotenv
from elasticsearch import Elasticsearch
import os

load_dotenv(override=True)

es = Elasticsearch(
    cloud_id=os.environ["ELASTIC_CLOUD_ID"],
    api_key=os.environ["ELASTIC_CLOUD_API_KEY"]
)
print(f'Client connected: {es.ping()}')

es.options(ignore_status=[404]).inference.delete(inference_id="jina-embeddings-v3-standard")
es.inference.put(
    task_type="text_embedding",
    inference_id="jina-embeddings-v3-standard",
    body={
        "service": "jinaai",
        "service_settings": {
            "api_key": os.environ["JINA_API_KEY"],
            "model_id": "jina-embeddings-v3",
            
        },
        "task_settings": {
            "late_chunking": False,
            "input_type": "search"
        }
    }
)
print("Standard inference endpoint created")
es.options(ignore_status=[404]).inference.delete(inference_id="jina-embeddings-v3-late")
es.inference.put(
    task_type="text_embedding",
    inference_id="jina-embeddings-v3-late",
    body={
        "service": "jinaai",
        "service_settings": {
            "api_key": os.environ["JINA_API_KEY"],
            "model_id": "jina-embeddings-v3",
            
        },
        "task_settings": {
            "late_chunking": True, 
            "input_type": "search"
        }
    }
)
print("Late inference endpoint created")

## Load Indices

In [ ]:
from late_chunking import faq, filing

faq.create_index(es)
faq.ingest(es, "assets/corpus_a_faq.json")
filing.create_index(es)
filing.ingest(es, "assets/corpus_b_filing.json")

## Retrieval & Measurement

In [ ]:
import json
from late_chunking import faq, filing

with open("assets/judgments.json") as f:
    judgments = json.load(f)

METRICS = {
    "MRR":  {"mean_reciprocal_rank": {"k": 5}},
    "NDCG": {"dcg":                  {"k": 5, "normalize": True}},
}

def build_std_requests(index, queries):
    return [
        {
            "id": str(i),
            "request": {
                "query": {"semantic": {"field": "embedding", "query": item['query']}},
                "size": 5
            },
            "ratings": [{"_index": index, "_id": r['chunk_id'], "rating": r['rating']} for r in item['ratings']]
        }
        for i, item in enumerate(queries)
    ]

def build_late_requests(es, index, queries):
    requests = []
    for i, item in enumerate(queries):
        vec = es.inference.inference(input=[item['query']], inference_id="jina-embeddings-v3-late")
        requests.append({
            "id": str(i),
            "request": {
                "knn": {"field": "embedding", "query_vector": vec['text_embedding'][0]['embedding'], "k": 5, "num_candidates": 50},
                "size": 5
            },
            "ratings": [{"_index": index, "_id": r['chunk_id'], "rating": r['rating']} for r in item['ratings']]
        })
    return requests

def eval_metrics(es, index, requests):
    return {
        name: es.rank_eval(index=index, requests=requests, metric=metric)['metric_score']
        for name, metric in METRICS.items()
    }

def report(label, es, std_index, late_index, queries):
    print(f"=== {label} ===\n")
    std_req  = build_std_requests(std_index, queries)
    late_req = build_late_requests(es, late_index, queries)
    std_scores  = eval_metrics(es, std_index,  std_req)
    late_scores = eval_metrics(es, late_index, late_req)
    print(f"{'Metric':<8} {'Standard':>10} {'Late':>10} {'Delta':>10}")
    print("-" * 40)
    for name in METRICS:
        s, l = std_scores[name], late_scores[name]
        print(f"{name:<8} {s:>10.4f} {l:>10.4f} {l - s:>+10.4f}")
    print()

report("Corpus A: Product FAQ", es, faq.FAQ_STANDARD_INDEX_NAME,         faq.FAQ_LATE_INDEX_NAME,             judgments['corpus_a'])
report("Corpus B: SEC 10-K Filings", es, filing.FILING_STANDARD_INDEX_NAME,   filing.FILING_LATE_INDEX_NAME,       judgments['corpus_b'])

## Destroy Environment

In [ ]:
%%bash
terraform -chdir=terraform destroy -auto-approve > /dev/null && echo "Done."
rm -f .env